# Getting Started with Fabric Governance

This notebook provides a quick start guide to the Fabric Governance toolkit.

## Quick Overview

This toolkit helps you manage Microsoft Fabric as an admin with the following capabilities:

1. **Audit Data Collection** - Collect and analyze activity logs
2. **Capacity Management** - View and manage Fabric capacities
3. **Workspace Migration** - Plan and execute workspace migrations
4. **Workspace Management** - Rename, create, and manage workspaces

## Prerequisites

1. Azure AD App Registration with required permissions
2. Credentials configured in `config/credentials.json`
3. Python packages installed (see `requirements.txt`)

In [ ]:
# Import required modules
import sys
sys.path.append('..')

from modules.fabric_auth import FabricAuth, load_credentials
from modules.fabric_client import FabricClient
import pandas as pd

## Step 1: Test Authentication

In [ ]:
# Load credentials and authenticate
try:
    credentials = load_credentials('../config/credentials.json')
    print("✓ Credentials loaded successfully")
    
    auth = FabricAuth(
        tenant_id=credentials['tenant_id'],
        client_id=credentials['client_id'],
        client_secret=credentials['client_secret']
    )
    
    token = auth.get_access_token()
    if token:
        print("✓ Authentication successful!")
        print("\nYou can now use the other notebooks to manage Fabric.")
        client = FabricClient(token)
    else:
        print("✗ Authentication failed")
        print("\nPlease check:")
        print("1. Your credentials are correct")
        print("2. Your app has the required API permissions")
        print("3. Admin consent has been granted")
        
except FileNotFoundError as e:
    print("✗ Credentials file not found")
    print("\nPlease:")
    print("1. Copy config/credentials.json.template to config/credentials.json")
    print("2. Fill in your Azure AD credentials")
except Exception as e:
    print(f"✗ Error: {e}")

## Step 2: Test API Access

In [ ]:
# Test basic API access
if token:
    print("Testing API access...\n")
    
    # Try to get workspaces
    workspaces = client.get_workspaces()
    if workspaces:
        print(f"✓ Successfully retrieved {len(workspaces)} workspaces")
        print("\nFirst 5 workspaces:")
        for i, ws in enumerate(workspaces[:5], 1):
            print(f"{i}. {ws.get('name', 'Unknown')} (ID: {ws.get('id', 'N/A')})")
    else:
        print("✗ Could not retrieve workspaces")
        print("\nPlease check your API permissions.")
    
    # Try to get capacities
    print("\n" + "="*50)
    capacities = client.get_capacities()
    if capacities:
        print(f"✓ Successfully retrieved {len(capacities)} capacities")
    else:
        print("⚠ No capacities found (this might be normal if you have no capacities)")

## Step 3: Quick Statistics

In [ ]:
# Display some quick statistics
if token and workspaces:
    print("=== Fabric Environment Overview ===")
    print(f"\nTotal Workspaces: {len(workspaces)}")
    
    # Workspace types
    ws_df = pd.DataFrame(workspaces)
    if 'type' in ws_df.columns:
        print("\nWorkspace Types:")
        print(ws_df['type'].value_counts())
    
    # Workspace states
    if 'state' in ws_df.columns:
        print("\nWorkspace States:")
        print(ws_df['state'].value_counts())
    
    # Capacity assignments
    if 'capacityId' in ws_df.columns:
        assigned = ws_df['capacityId'].notna().sum()
        unassigned = ws_df['capacityId'].isna().sum()
        print(f"\nCapacity Assignments:")
        print(f"  Assigned to capacity: {assigned}")
        print(f"  Not assigned: {unassigned}")

## Next Steps

If authentication was successful, you can now use the other notebooks:

1. **01_audit_data_collection.ipynb** - Collect audit logs and activity data
2. **02_capacity_management.ipynb** - Manage Fabric capacities
3. **03_workspace_migration.ipynb** - Plan and execute workspace migrations
4. **04_workspace_management.ipynb** - Rename and manage workspaces

## Troubleshooting

If you encounter issues:

1. **Authentication fails**: Check your credentials and API permissions
2. **No data returned**: Verify you have admin access to the tenant
3. **Module not found**: Ensure you're running from the notebooks directory
4. **Import errors**: Install dependencies with `pip install -r requirements.txt`